# Healthcare Revenue Cycle Management Agent with Google ADK

## Business Problem

Healthcare teams often check insurance, choose billing codes, send claims, and handle rejected claims by hand. This work can be slow. Missing data or a wrong code can delay payment or cause lost revenue.

## How This System Solves the Problem

This system puts the main billing steps in one Google ADK workflow. It reads claim data, checks insurance, suggests ICD-10 codes, sends a test claim, finds problems, tries coding again when needed, and writes an appeal for a rejected claim. It also saves a short claim summary.

## Notebook Steps

1. Run the package install cell.
2. Add your Google Cloud project ID.
3. Sign in to Google Cloud.
4. Run the cells from top to bottom.
5. Check the test claim results.
6. Run the Cloud Run cells only when you are ready to deploy.

> This is a technical prototype. Do not use real patient data or real billing decisions.

## 1. Install packages with uv

Make sure `uv` is installed. This cell installs the notebook packages.

In [ ]:
!uv pip install --system -q "google-adk[a2a]>=2.3.0,<3.0.0" "mcp>=1.27.0,<2.0.0" "pydantic>=2.11.0,<3.0.0"

## 2. Google Cloud setup

Change the project ID. Colab opens a Google login screen. For local Jupyter, run `gcloud auth application-default login` first.

In [ ]:
import os

PROJECT_ID = "your-project-id"
REGION = "us-central1"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = REGION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["MODEL_NAME"] = "gemini-2.5-flash"
os.environ["CLAIM_API_DELAY_SECONDS"] = "0.1"

if PROJECT_ID == "your-project-id":
    print("Set PROJECT_ID before calling Gemini or deploying.")

## 3. Create the complete ADK agent

The dynamic workflow uses a normal loop. Gemini failures use local fallbacks.

In [ ]:
from pathlib import Path

Path("rcm_agent").mkdir(exist_ok=True)
Path("rcm_agent/__init__.py").write_text("from . import agent\n", encoding="utf-8")

In [ ]:
%%writefile rcm_agent/agent.py
import asyncio
import os
import re
import sqlite3
from pathlib import Path
from typing import Literal
from uuid import uuid4

from google.adk import Agent, Context, Workflow
from google.adk.workflow import node
from pydantic import BaseModel, Field


MAX_RETRIES = 2
ALLOWED_CODES = {"E11.9", "I10", "J10.1", "R50.9", "R69"}


class ClaimRequest(BaseModel):
    patient_id: str = Field(min_length=1, max_length=40)
    insurance_number: str = Field(min_length=3, max_length=40)
    visit_reason: str = Field(min_length=3, max_length=1000)
    history: list[str] = Field(default_factory=list)
    retry_count: int = 0


class VerifiedClaim(ClaimRequest):
    insurance_verified: bool


class CodedClaim(VerifiedClaim):
    codes: list[str] = Field(default_factory=list)
    needs_human_review: bool = False
    coding_source: Literal["gemini", "heuristic"] = "gemini"


class ClaimDecision(CodedClaim):
    status: Literal["Submitted", "Rejected", "NeedsReview"]
    claim_id: str | None = None
    operation_id: str | None = None
    denial_reason: str | None = None


class AuditedClaim(ClaimDecision):
    audit_issues: list[str] = Field(default_factory=list)
    issue_score: int = 0
    retry_needed: bool = False


class ClaimResult(AuditedClaim):
    appeal: str | None = None
    summary: str
    context_summary: str = ""
    memory_id: str | None = None


def add_history(history: list[str], message: str) -> list[str]:
    return [*history, message]


def parse_claim_fallback(text: str) -> ClaimRequest:
    patterns = {
        "patient_id": r"patient\s*id\s*:\s*(.+)",
        "insurance_number": r"insurance(?:\s*number)?\s*:\s*(.+)",
        "visit_reason": r"visit\s*reason\s*:\s*(.+)",
    }
    fields = {}
    for name, pattern in patterns.items():
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            fields[name] = match.group(1).strip()

    return ClaimRequest(
        patient_id=fields.get("patient_id", "UNKNOWN"),
        insurance_number=fields.get("insurance_number", "UNKNOWN"),
        visit_reason=fields.get("visit_reason", text.strip() or "Unknown visit reason"),
        history=["Intake used the local fallback parser"],
    )


def verify_insurance(claim: ClaimRequest) -> VerifiedClaim:
    number = claim.insurance_number.strip().upper()
    verified = number.startswith(("AET", "BLU"))
    return VerifiedClaim(
        **claim.model_dump(exclude={"insurance_number", "history"}),
        insurance_number=number,
        insurance_verified=verified,
        history=add_history(claim.history, f"Insurance verified: {verified}"),
    )


def heuristic_coding(claim: VerifiedClaim) -> CodedClaim:
    text = claim.visit_reason.lower()
    codes = []
    if "diabetes" in text:
        codes.append("E11.9")
    if "hypertension" in text or "high blood pressure" in text:
        codes.append("I10")
    if "flu" in text or "influenza" in text:
        codes.append("J10.1")
    elif "fever" in text:
        codes.append("R50.9")

    needs_review = not codes
    if needs_review:
        codes = ["R69"]

    return CodedClaim(
        **claim.model_dump(exclude={"history"}),
        codes=codes,
        needs_human_review=needs_review,
        coding_source="heuristic",
        history=add_history(claim.history, f"Heuristic coding returned: {codes}"),
    )


def validate_codes(claim: CodedClaim) -> CodedClaim:
    original = [code.upper().strip() for code in claim.codes]
    codes = list(dict.fromkeys(code for code in original if code in ALLOWED_CODES))
    needs_review = claim.needs_human_review or not codes or "R69" in codes
    return claim.model_copy(
        update={
            "codes": codes,
            "needs_human_review": needs_review,
            "history": add_history(claim.history, f"Validated codes: {codes or 'none'}"),
        }
    )


async def submit_claim(claim: CodedClaim) -> ClaimDecision:
    operation_id = f"OP-{uuid4().hex[:10].upper()}"
    await asyncio.sleep(float(os.getenv("CLAIM_API_DELAY_SECONDS", "0.1")))
    values = claim.model_dump(exclude={"history"})
    history = add_history(claim.history, f"Claim operation completed: {operation_id}")

    if not claim.insurance_verified:
        return ClaimDecision(
            **values,
            history=history,
            status="Rejected",
            operation_id=operation_id,
            denial_reason="Insurance could not be verified",
        )
    if not claim.codes:
        return ClaimDecision(
            **values,
            history=history,
            status="NeedsReview",
            operation_id=operation_id,
            denial_reason="No valid diagnosis code was found",
        )
    if claim.needs_human_review:
        return ClaimDecision(
            **values,
            history=history,
            status="NeedsReview",
            operation_id=operation_id,
            denial_reason="A coding specialist must review this claim",
        )

    claim_id = f"CLM-{uuid4().hex[:12].upper()}"
    return ClaimDecision(
        **values,
        history=add_history(history, f"Claim submitted: {claim_id}"),
        status="Submitted",
        claim_id=claim_id,
        operation_id=operation_id,
    )


def audit_claim(claim: ClaimDecision) -> AuditedClaim:
    issues = []
    if not claim.insurance_verified:
        issues.append("Insurance is not verified")
    if not claim.codes:
        issues.append("No diagnosis code was found")
    if claim.needs_human_review:
        issues.append("A human coding review is required")
    if claim.status == "Rejected":
        issues.append("The claim was rejected")

    retry_needed = not claim.codes and claim.retry_count < MAX_RETRIES
    score = len(issues)
    return AuditedClaim(
        **claim.model_dump(exclude={"history"}),
        history=add_history(
            claim.history,
            f"Audit score: {score}; retry needed: {retry_needed}",
        ),
        audit_issues=issues,
        issue_score=score,
        retry_needed=retry_needed,
    )


def prepare_retry(claim: AuditedClaim) -> VerifiedClaim:
    retry_count = claim.retry_count + 1
    return VerifiedClaim(
        patient_id=claim.patient_id,
        insurance_number=claim.insurance_number,
        visit_reason=claim.visit_reason,
        insurance_verified=claim.insurance_verified,
        retry_count=retry_count,
        history=add_history(claim.history, f"Starting coding retry {retry_count}"),
    )


def compact_context(history: list[str], max_chars: int = 500) -> str:
    return " | ".join(history)[-max_chars:]


def fallback_result(claim: AuditedClaim) -> ClaimResult:
    appeal = None
    if claim.status == "Rejected":
        appeal = (
            "Dear Medical Review Team,\n\n"
            f"Please review the denied claim for patient {claim.patient_id}. "
            f"The denial reason is: {claim.denial_reason or 'Unknown reason'}. "
            f"The diagnosis codes were: {', '.join(claim.codes) or 'None'}.\n\n"
            "Please reconsider the claim using the available visit record.\n\n"
            "Sincerely,\nRCM Billing Team"
        )

    return ClaimResult(
        **claim.model_dump(),
        appeal=appeal,
        summary=(
            f"Claim status: {claim.status}. Codes: {', '.join(claim.codes) or 'None'}. "
            f"Audit score: {claim.issue_score}."
        ),
        context_summary=compact_context(claim.history),
    )


def save_memory(result: ClaimResult) -> ClaimResult:
    path = Path(os.getenv("MEMORY_DB_PATH", "claim_memory.db"))
    memory_id = f"MEM-{result.operation_id or uuid4().hex[:10]}"
    with sqlite3.connect(path) as connection:
        connection.execute(
            """
            CREATE TABLE IF NOT EXISTS claim_memory (
                memory_id TEXT PRIMARY KEY,
                patient_id TEXT,
                claim_id TEXT,
                status TEXT,
                codes TEXT,
                summary TEXT,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP
            )
            """
        )
        connection.execute(
            """
            INSERT OR REPLACE INTO claim_memory
                (memory_id, patient_id, claim_id, status, codes, summary)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                memory_id,
                result.patient_id,
                result.claim_id,
                result.status,
                ",".join(result.codes),
                compact_context(result.history),
            ),
        )
    return result.model_copy(update={"memory_id": memory_id})


model = os.getenv("MODEL_NAME", "gemini-2.5-flash")

intake_agent = Agent(
    name="intake_agent",
    model=model,
    instruction=(
        "Read the claim. Return patient ID, insurance number, and visit reason. "
        "Copy history and retry count. Do not add facts."
    ),
    output_schema=ClaimRequest,
)

coding_agent = Agent(
    name="coding_agent",
    model=model,
    input_schema=VerifiedClaim,
    instruction=(
        "Copy all input fields. Use only E11.9 for diabetes, I10 for high blood "
        "pressure, J10.1 for flu, R50.9 for fever, or R69 when unclear. "
        "Set coding_source to gemini and mark unclear results for human review."
    ),
    output_schema=CodedClaim,
)

appeal_agent = Agent(
    name="appeal_agent",
    model=model,
    input_schema=AuditedClaim,
    instruction=(
        "Copy all input fields. Write a short appeal only for a rejected claim. "
        "Use only given facts. Add a short summary and compact context summary."
    ),
    output_schema=ClaimResult,
)


@node
def insurance_node(claim: ClaimRequest):
    return verify_insurance(claim)


@node
def heuristic_node(claim: VerifiedClaim):
    return heuristic_coding(claim)


@node
def validation_node(claim: CodedClaim):
    return validate_codes(claim)


@node
async def submission_node(claim: CodedClaim):
    return await submit_claim(claim)


@node
def audit_node(claim: ClaimDecision):
    return audit_claim(claim)


@node
def retry_node(claim: AuditedClaim):
    return prepare_retry(claim)


@node
def memory_node(result: ClaimResult):
    return save_memory(result)


@node(name="rcm_pipeline", rerun_on_resume=True)
async def rcm_pipeline(ctx: Context, user_request: str) -> ClaimResult:
    try:
        claim = await ctx.run_node(intake_agent, user_request)
    except Exception:
        claim = parse_claim_fallback(user_request)

    verified = await ctx.run_node(insurance_node, claim)

    while True:
        try:
            coded = await ctx.run_node(coding_agent, verified)
        except Exception:
            coded = await ctx.run_node(heuristic_node, verified)

        checked = await ctx.run_node(validation_node, coded)
        decision = await ctx.run_node(submission_node, checked)
        audited = await ctx.run_node(audit_node, decision)

        if not audited.retry_needed:
            break
        verified = await ctx.run_node(retry_node, audited)

    try:
        result = await ctx.run_node(appeal_agent, audited)
    except Exception:
        result = fallback_result(audited)

    return await ctx.run_node(memory_node, result)


root_agent = Workflow(
    name="rcm_agent",
    description="Reduces claim errors through intake, coding, audit, retry, and appeal.",
    edges=[("START", rcm_pipeline)],
)


## 4. Check the fixed business rules

This cell does not call Gemini.

In [ ]:
import asyncio

from rcm_agent.agent import (
    ClaimRequest,
    audit_claim,
    heuristic_coding,
    submit_claim,
    validate_codes,
    verify_insurance,
)

request = ClaimRequest(
    patient_id="P001",
    insurance_number="AET123456",
    visit_reason="Type 2 diabetes and high blood pressure",
)
verified = verify_insurance(request)
coded = heuristic_coding(verified)
decision = await submit_claim(validate_codes(coded))
audited = audit_claim(decision)

assert decision.status == "Submitted"
assert set(decision.codes) == {"E11.9", "I10"}
assert audited.issue_score == 0
print("Business rule test passed.")

## 5. Run the ADK workflow

A fresh in-memory session is created for each example.

In [ ]:
from uuid import uuid4

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from rcm_agent.agent import root_agent


async def run_claim(message: str) -> str:
    app_name = "rcm_notebook"
    user_id = "test_user"
    session_id = uuid4().hex
    sessions = InMemorySessionService()

    await sessions.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id,
    )
    runner = Runner(
        agent=root_agent,
        app_name=app_name,
        session_service=sessions,
    )
    content = types.Content(role="user", parts=[types.Part(text=message)])
    final_text = ""

    events = runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content,
    )
    async for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text or ""

    print(final_text)
    return final_text

In [ ]:
valid_claim = """Patient ID: P001
Insurance number: AET123456
Visit reason: Type 2 diabetes and high blood pressure
"""

valid_result = await run_claim(valid_claim)

In [ ]:
rejected_claim = """Patient ID: P002
Insurance number: XYZ999
Visit reason: High fever
"""

rejected_result = await run_claim(rejected_claim)

## 6. A2A service

ADK creates the A2A agent card and protocol routes.

In [ ]:
%%writefile a2a_server.py
import os
import uvicorn
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from rcm_agent.agent import root_agent

port = int(os.getenv("PORT", "8001"))
app = to_a2a(root_agent, port=port)

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=port)

Run the A2A service in a terminal with:

`uv run uvicorn a2a_server:app --host 0.0.0.0 --port 8001`

## 7. MCP service

This server exposes four tools through the MCP protocol.

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP

from rcm_agent.agent import CodedClaim, submit_claim as send_claim, validate_codes


mcp = FastMCP("rcm-tools", stateless_http=True, json_response=True)


@mcp.tool()
def verify_insurance(insurance_number: str) -> dict:
    """Check a prototype insurance number."""
    number = insurance_number.strip().upper()
    return {"insurance_number": number, "verified": number.startswith(("AET", "BLU"))}


@mcp.tool()
def search_code_catalog(query: str) -> list[dict]:
    """Search the diagnosis code catalog."""
    catalog = {
        "E11.9": "Type 2 diabetes without complications",
        "I10": "High blood pressure",
        "J10.1": "Flu with breathing symptoms",
        "R50.9": "Fever, cause not known",
        "R69": "Illness, cause not clear",
    }
    words = query.lower().split()
    results = [
        {"code": code, "description": text}
        for code, text in catalog.items()
        if any(word in text.lower() for word in words)
    ]
    return results or [{"code": "R69", "description": catalog["R69"]}]


@mcp.tool()
def calculate_issue_score(issues: list[str]) -> dict:
    """Count audit issues."""
    return {"issue_score": len(issues), "issues": issues}


@mcp.tool()
async def submit_claim(codes: list[str], verified: bool) -> dict:
    """Send a test claim with diagnosis codes."""
    claim = CodedClaim(
        patient_id="MCP-TEST",
        insurance_number="MCP-TEST",
        visit_reason="MCP tool call",
        insurance_verified=verified,
        codes=codes,
        needs_human_review=not codes,
        coding_source="heuristic",
    )
    result = await send_claim(validate_codes(claim))
    return result.model_dump()


if __name__ == "__main__":
    mcp.run(transport="streamable-http")


Run the MCP server in a terminal with:

`uv run python mcp_server.py`

The default Streamable HTTP endpoint is `http://localhost:8000/mcp`.

## 8. Deploy to Cloud Run

In [ ]:
if PROJECT_ID == "your-project-id":
    raise ValueError("Set PROJECT_ID before deployment.")

!gcloud config set project $PROJECT_ID
!gcloud services enable run.googleapis.com aiplatform.googleapis.com cloudbuild.googleapis.com artifactregistry.googleapis.com --project=$PROJECT_ID

In [ ]:
!adk deploy cloud_run \
  --project=$PROJECT_ID \
  --region=$REGION \
  --service_name=rcm-adk-service \
  --app_name=rcm_agent \
  --with_ui \
  ./rcm_agent \
  -- \
  --no-allow-unauthenticated \
  --set-env-vars=GOOGLE_GENAI_USE_VERTEXAI=True,GOOGLE_CLOUD_PROJECT=$PROJECT_ID,GOOGLE_CLOUD_LOCATION=$REGION

## TODO

1. Connect an insurance eligibility API and a claim clearinghouse in a safe test environment.
2. Use an approved ICD-10 and CPT data source. Require a medical coder to approve codes before a claim is sent.
3. Move sessions and memory to Cloud SQL. Add IAM, Secret Manager, and encrypted audit logs.

## Safety checklist

- Keep all services private.
- Use Cloud SQL or Agent Runtime instead of local SQLite.
- Use real eligibility, code catalog, and clearinghouse APIs.
- Require human approval before sending a claim.
- Add access control, audit retention, encryption, monitoring, and deletion rules.
- Confirm legal and Google Cloud BAA requirements before using protected health data.